### Data Preprocessing

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# Load dataset
df1 = pd.read_csv('Data_2004_2013.csv')

# Set the timestamp column as the index
df1['TimeStamp_1'] = pd.to_datetime(df1['TimeStamp_1'], format='mixed')
df1.set_index('TimeStamp_1', inplace=True)

# Train-Test split
split1 = pd.to_datetime('2007-10-31 23:59:00')
split2 = pd.to_datetime('2008-10-31 23:59:00')
split3 = pd.to_datetime('2009-10-31 23:59:00')
df_train = df1[(df1.index < split1) | (df1.index > split3)]
df_test = df1[(df1.index > split1) & (df1.index < split2)]
df_val = df1[(df1.index > split2) & (df1.index < split3)]

# Data normalization
scaled_wave = MinMaxScaler()
scaled_wave = scaled_wave.fit(df1[(df1.index < split1) | (df1.index > split2)][['H']])

### Model Evaluation

In [2]:
rmse = lambda a,b: np.sqrt(mean_squared_error(a,b))

def evaluate(df, name, pred):
    # Split windy and calm seasons
    month = df.index.month
    df_w = df[(month >= 11) | (month <= 3)]
    df_c = df[(month >= 4) & (month <= 10)]
    
    y_pred = scaled_wave.inverse_transform(df[pred].values.reshape(-1, 1)).flatten()
    y_pred_w = y_pred[(month >= 11) | (month <= 3)]
    y_pred_c = y_pred[(month >= 4) & (month <= 10)]

    print(f'{name} windy/calm/overall RMSE: '
          f'{rmse(df_w['H'][6:], y_pred_w[6:]):.3f} / '
          f'{rmse(df_c['H'], y_pred_c):.3f} / '
          f'{rmse(df['H'][6:], y_pred[6:]):.3f}')
    
    y_true_w = df_w['H'][6:].reset_index(drop=True)
    y_pred_w = pd.Series(y_pred_w[6:]).reset_index(drop=True)
    y_true_c = df_c['H'].reset_index(drop=True)
    y_pred_c = pd.Series(y_pred_c).reset_index(drop=True)
    y_true = df['H'][6:].reset_index(drop=True)
    y_pred = pd.Series(y_pred[6:]).reset_index(drop=True)

    mask_w = y_true_w >= 4
    mask_c = y_true_c >= 4
    mask = y_true >= 4

    print(f'{name} windy/calm/overall RMSE (H≥4): '
          f'{rmse(y_true_w[mask_w], y_pred_w[mask_w]):.3f} / '
          f'{rmse(y_true_c[mask_c], y_pred_c[mask_c]):.3f} / '
          f'{rmse(y_true[mask], y_pred[mask]):.3f}')
    print()

print('General wave height regressor')
evaluate(df_train, 'Train', 'H_pred0')
evaluate(df_val, 'Val', 'H_pred0')
evaluate(df_test, 'Test', 'H_pred0')

print('Big-wave-focused regressor')
evaluate(df_train, 'Train', 'H_pred1')
evaluate(df_val, 'Val', 'H_pred1')
evaluate(df_test, 'Test', 'H_pred1')

General wave height regressor
Train windy/calm/overall RMSE: 0.959 / 0.663 / 0.799
Train windy/calm/overall RMSE (H≥4): 1.518 / 2.160 / 1.636

Val windy/calm/overall RMSE: 0.929 / 0.635 / 0.770
Val windy/calm/overall RMSE (H≥4): 1.523 / 2.400 / 1.679

Test windy/calm/overall RMSE: 0.829 / 0.676 / 0.743
Test windy/calm/overall RMSE (H≥4): 1.617 / 2.706 / 1.873

Big-wave-focused regressor
Train windy/calm/overall RMSE: 1.657 / 1.000 / 1.312
Train windy/calm/overall RMSE (H≥4): 0.969 / 1.834 / 1.149

Val windy/calm/overall RMSE: 1.498 / 0.920 / 1.194
Val windy/calm/overall RMSE (H≥4): 0.876 / 1.870 / 1.079

Test windy/calm/overall RMSE: 1.514 / 0.960 / 1.221
Test windy/calm/overall RMSE (H≥4): 0.956 / 2.067 / 1.245



In [3]:
# Adjust the shape of Sigmoid function
def adjusted_probs(p, shift):
    p = np.clip(p, 1e-10, 1 - 1e-10)
    logit = np.log(p / (1 - p))
    return 1 / (1 + np.exp(-(logit - shift)))

# Search the best shift parameter
def search_shifts(df):
    best_shift, lowest_rmse = None, float('inf')
    for shift in np.arange(-5, 5.01, 0.01):
        probs = adjusted_probs(df['Spike_prob'], shift)
        y_pred = (1 - probs) * df['H_pred0'] + probs * df['H_pred1']
        y_pred_raw = scaled_wave.inverse_transform(y_pred.values.reshape(-1, 1))
        if rmse(df['H'], y_pred_raw) < lowest_rmse:
            best_shift, lowest_rmse = shift, rmse(df['H'], y_pred_raw)
    return best_shift

# Best shifts in the windy season
df = df_val[(df_val.index.month <= 3) | (df_val.index.month >= 11)]
best_shifts = [search_shifts(df)] * 12

# Best shifts in the calm season
for i in range(3, 10):
    df = df_val[df_val.index.month == i+1]
    best_shifts[i] = search_shifts(df)

print('Best shifts:', ', '.join(f'{x:.3f}' for x in best_shifts))

Best shifts: 4.130, 4.130, 4.130, 5.000, 5.000, 5.000, 2.350, 4.500, -2.680, -2.540, 4.130, 4.130


In [4]:
month = df_test.index.month

for i in range(12):
    month_mask = (month == i+1)
    df_test.loc[month_mask, 'Spike_prob'] = adjusted_probs(df_test.loc[month_mask, 'Spike_prob'], best_shifts[i])

# Windy season
df_test1 = df_test[(month >= 11) | (month <= 3)]
y_pred1 = (1 - df_test1['Spike_prob']) * df_test1['H_pred0'] + df_test1['Spike_prob'] * df_test1['H_pred1']
y_pred1_raw = scaled_wave.inverse_transform(y_pred1.values.reshape(-1, 1)).flatten()

# Calm season
df_test2 = df_test[(month >= 4) & (month <= 10)]
y_pred2 = (1 - df_test2['Spike_prob']) * df_test2['H_pred0'] + df_test2['Spike_prob'] * df_test2['H_pred1']
y_pred2_raw = scaled_wave.inverse_transform(y_pred2.values.reshape(-1, 1)).flatten()

# Year-round
y_pred = (1 - df_test['Spike_prob']) * df_test['H_pred0'] + df_test['Spike_prob'] * df_test['H_pred1']
y_pred_raw = scaled_wave.inverse_transform(y_pred.values.reshape(-1, 1)).flatten()

print('Stacked model')
print(f'Test windy/calm/overall RMSE: '
      f'{rmse(df_test1['H'], y_pred1_raw):.3f} / '
      f'{rmse(df_test2['H'], y_pred2_raw):.3f} / '
      f'{rmse(df_test['H'], y_pred_raw):.3f}')

Stacked model
Test windy/calm/overall RMSE: 0.800 / 0.640 / 0.711
